# Task A KIM batch #2 - zero-knowledge LI control (gated)

**This notebook is gated on batch 1.** It first re-reads the batch-1 probability
dumps and checks the pre-registered KIM-I success bar. If KIM-I did not clear
that bar, the notebook stops and spends no GPU time.

**Why a zero-knowledge control.** KIM-I adds a local-inference block *and*
WordNet features. The block itself could help even if `R_ij` is noise. `KIM0`
builds the exact same last-block local-inference block but feeds an all-zero
knowledge matrix (`--kim-zero-knowledge`). Comparing:

| comparison | what it isolates |
|---|---|
| `KIM0` vs `A160r` | the local-inference architecture, no knowledge |
| `KIMI` vs `KIM0` | the five WordNet features, same architecture |

`KIM0` is an architecture control, not a KIM arm. It runs only if batch 1 passes.

**KIM-A is not implemented and is deliberately deferred.** The co-attention
("A") component of KIM requires a different attention construction; this repo
only has KIM-I. Do not fabricate a KIM-A arm. Stage it only after KIM-I shows a
positive, knowledge-specific gain that survives `KIM0`.

**Checklist.** Same data, GPU T4 x2, Internet On. If this is a fresh session,
attach the batch-1 output as a Kaggle dataset so the gate cell can find
`KIMI_s*_val_probs.npy` and `A160r_s*_val_probs.npy`. Runtime for `KIM0`: about
2-3 h for three seeds.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '70c3b660fb5bc6b53a35e48c65cdd000b9ac84ea'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)

## 1. Data and split-42 validation frame

The frame is rebuilt deterministically, so its row order matches batch 1. The
gate cell below expects the batch-1 arrays and probability dumps; if you attach
a dataset, put its files anywhere under `/kaggle/input/**`. You can also set
`BATCH1_DIR` to the directory holding them.

In [ ]:
import glob
import os
import sys

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
tokenizer = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token

df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738, f'unexpected val size: {len(labs)}'
assert int(labs.sum()) == 276, f'unexpected positive count: {int(labs.sum())}'
assert df_val['yt_title'].nunique() == 56, df_val['yt_title'].nunique()

np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
np.save('weights/val_ids.npy', df_val['StereoQueerEval_id'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads.npy', quads)
print('val:', len(df_val), 'rows | pos', int(labs.sum()), '| clusters',
      df_val['yt_title'].nunique(), flush=True)
print('  quadrants', dict(pd.Series(quads).value_counts()), flush=True)

## 2. Batch-1 gate

The gate uses the same paired video-cluster bootstrap as batch 1. `mean best`
uses each run's history if the CSV was attached; otherwise it falls back to the
reload macro-F1 from the saved probabilities (printed clearly). The gate is the
batch-1 pre-registered rule: mean >= +0.5 pt, ensemble >= +0.5 pt, CI low > 0,
and no language worse by more than 2 pt.

In [ ]:
import json

from sklearn.metrics import f1_score, roc_auc_score

BATCH1_DIR = os.environ.get('BATCH1_DIR', 'weights').strip() or 'weights'
print('batch-1 directory:', BATCH1_DIR, flush=True)


def find_artifact(name):
    direct = os.path.join(BATCH1_DIR, name)
    if os.path.exists(direct):
        return direct
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None


def load_batch1_probs(tag):
    p = find_artifact(f'{tag}_val_probs.npy')
    if p is None:
        return None, None
    arr = np.load(p)
    if arr.shape != (len(labs), 2):
        print(f'!! {p}: shape {arr.shape}, expected {(len(labs), 2)}', flush=True)
        return None, None
    return arr, p


def hist_best(tag):
    p = find_artifact(f'{tag}_history.csv')
    if p is None:
        return None
    h = pd.read_csv(p)
    return float(h['val_macro_f1'].max())


def macro_f1_binary(y, pred):
    return float(f1_score(y, pred, labels=[0, 1], average='macro', zero_division=0))


videos = np.load('weights/val_videos.npy', allow_pickle=True).astype(str)
clusters = np.unique(videos)
groups = [np.flatnonzero(videos == c) for c in clusters]
print(f'{len(clusters)} video clusters | {len(labs)} rows', flush=True)


def paired_cluster_bootstrap(y, probs_a, probs_b, n_boot=5000, seed=12345):
    rng = np.random.default_rng(seed)
    obs_a = macro_f1_binary(y, probs_a.argmax(1))
    obs_b = macro_f1_binary(y, probs_b.argmax(1))
    diffs = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        pick = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        diffs[i] = (macro_f1_binary(y[idx], probs_a[idx].argmax(1))
                    - macro_f1_binary(y[idx], probs_b[idx].argmax(1)))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2.0 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))
    return dict(observed=obs_a - obs_b, a=obs_a, b=obs_b,
                bootstrap_mean=float(diffs.mean()),
                ci_low=float(lo), ci_high=float(hi),
                p_value=float(min(1.0, p)), n_boot=n_boot,
                n_clusters=len(groups))


langs = np.load('weights/val_langs.npy', allow_pickle=True).astype(str)


def per_language(probs):
    out = {}
    for lang in sorted(set(langs.tolist())):
        msk = langs == lang
        m = tx.metrics_from(probs[msk].argmax(1).tolist(), labs[msk].tolist())
        out[lang] = float(m['macro_f1'])
    return out


SEEDS = (42, 43, 44)
probs_by_tag = {}
for arm in ('A160r', 'KIMI'):
    for s in SEEDS:
        p, path = load_batch1_probs(f'{arm}_s{s}')
        if p is None:
            print(f'!! missing {arm}_s{s}_val_probs.npy', flush=True)
        else:
            probs_by_tag[f'{arm}_s{s}'] = p
            print(f'  {arm}_s{s}: {path}', flush=True)

PROCEED = False
gate = {}
if all(f'{arm}_s{s}' in probs_by_tag for arm in ('A160r', 'KIMI') for s in SEEDS):
    ens = {}
    for arm in ('A160r', 'KIMI'):
        tags = [f'{arm}_s{s}' for s in SEEDS]
        ens[arm] = np.mean([probs_by_tag[t] for t in tags], axis=0)
        hb = [hist_best(t) for t in tags]
        mean_hist = None if any(v is None for v in hb) else float(np.mean(hb))
        mean_reload = float(np.mean([
            macro_f1_binary(labs, probs_by_tag[t].argmax(1)) for t in tags
        ]))
        gate[arm] = dict(
            mean_best=mean_hist if mean_hist is not None else mean_reload,
            mean_best_source='history' if mean_hist is not None else 'reload_fallback',
            mean_reload=mean_reload,
            ensemble_f1=macro_f1_binary(labs, ens[arm].argmax(1)),
            ensemble_auc=float(roc_auc_score(labs, ens[arm][:, 1])),
            languages=per_language(ens[arm]),
        )
    dm = gate['KIMI']['mean_best'] - gate['A160r']['mean_best']
    de = gate['KIMI']['ensemble_f1'] - gate['A160r']['ensemble_f1']
    boot_gate = paired_cluster_bootstrap(labs, ens['KIMI'], ens['A160r'],
                                         n_boot=5000, seed=12345)
    lang_delta = {k: gate['KIMI']['languages'][k] - gate['A160r']['languages'][k]
                  for k in gate['A160r']['languages']}
    min_lang_delta = min(lang_delta.values())
    PASS_MEAN = dm >= 0.005
    PASS_ENS = de >= 0.005
    PASS_CI = boot_gate['ci_low'] > 0.0
    PASS_LANG = min_lang_delta >= -0.02
    PROCEED = bool(PASS_MEAN and PASS_ENS and PASS_CI and PASS_LANG)
    print('\n--- BATCH-1 GATE ---', flush=True)
    for arm in ('A160r', 'KIMI'):
        g = gate[arm]
        print(f"{arm}: mean best {g['mean_best']:.4f} ({g['mean_best_source']}) | "
              f"mean reload {g['mean_reload']:.4f} | ensemble {g['ensemble_f1']:.4f} | "
              f"AUC {g['ensemble_auc']:.4f}", flush=True)
        print(f"  languages {g['languages']}", flush=True)
    print(f'KIMI - A160r: mean {dm:+.4f} | ensemble {de:+.4f} | '
          f'CI [{boot_gate["ci_low"]:+.4f}, {boot_gate["ci_high"]:+.4f}] | '
          f'p {boot_gate["p_value"]:.4f}', flush=True)
    print(f'checks: mean {PASS_MEAN} | ensemble {PASS_ENS} | CI {PASS_CI} | '
          f'language {PASS_LANG} -> PROCEED {PROCEED}', flush=True)
    if PROCEED:
        print('Batch-1 gate passed. KIM0 (zero-knowledge LI) will run.', flush=True)
    else:
        print('Batch-1 gate FAILED. KIM0 is skipped; KIM-A is not justified.', flush=True)
    gate['delta'] = dict(mean=float(dm), ensemble=float(de),
                         language={k: float(v) for k, v in lang_delta.items()},
                         min_language=float(min_lang_delta))
    gate['bootstrap'] = boot_gate
    gate['checks'] = dict(mean=bool(PASS_MEAN), ensemble=bool(PASS_ENS),
                          ci=bool(PASS_CI), language=bool(PASS_LANG))
    gate['proceed'] = bool(PROCEED)
    with open('weights/taskA_kim_batch2_gate.json', 'w', encoding='utf-8') as fh:
        json.dump(gate, fh, indent=2)
else:
    print('Batch-1 artifacts incomplete; gate cannot be evaluated. '
          'Attach the batch-1 output or set BATCH1_DIR.', flush=True)
    print('PROCEED = False', flush=True)

## 3. KIM0 run list and helpers

`KIM0` is exactly the KIM-I architecture with `--kim-zero-knowledge`, so the
knowledge tensor is all zeros but the local-inference block, gate, projection,
and extra parameters are all present. Same seeds 42/43/44 and split 42. Plain
A160 recipe otherwise.

KIM-A is not in this list. There is no KIM-A implementation in the pinned code,
and this batch is only the control for KIM-I.

In [ ]:
BASE = dict(
    dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
    backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
    pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
    max_len=160, video_max_len=256, no_amp=False,
    backbone=model_xattn.DEFAULT_BACKBONE, knowledge_dim=5,
)

RUNS = [dict(BASE, tag=f'KIM0_s{s}', seed=s, split_seed=42,
             kim_local=True, zero_knowledge=True) for s in (42, 43, 44)]


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    probs = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(hist) and os.path.exists(out):
        print(f"-- skip (checkpoint + history exist): {r['tag']}", flush=True)
        return True
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"--kim-local --kim-zero-knowledge --kim-knowledge-dim {r['knowledge_dim']} "
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist} --probs-out {probs}"
    )
    if r['no_amp']:
        cmd += ' --no-amp'
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    gate_txt = ''
    if 'kim_gate' in h.columns:
        gate_txt = (f" | kim_gate best {float(h.loc[bi, 'kim_gate']):+.5f} "
                    f"last {float(h['kim_gate'].iloc[-1]):+.5f} "
                    f"max {float(h['kim_gate'].max()):+.5f}")
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {float(h.loc[bi, 'val_macro_f1']):.4f} | "
          f"acc {float(h.loc[bi, 'val_acc']):.4f}{gate_txt} | {len(h)} epochs", flush=True)
    return True


def dump_probs(r):
    out = f"weights/{r['tag']}.pt"
    p = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(p) and np.load(p).shape == (len(labs), 2):
        print(f'-- skip probs: {r["tag"]}', flush=True)
        return
    if not os.path.exists(out):
        print(f'-- no checkpoint for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        num_classes=2, kim_local=True, knowledge_dim=r['knowledge_dim'],
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(
        df_val, tokenizer, 32, r['max_len'], r['video_max_len'],
        with_knowledge=True, knowledge_cache=None,
        knowledge_dim=r['knowledge_dim'], zero_knowledge=True,
    )
    o = tx.collect(net, loader, dev, use_amp=True, with_knowledge=True)
    np.save(p, o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)
print('PROCEED:', PROCEED, flush=True)

## 4. Train KIM0 (only if the gate passed)

If `PROCEED` is false, this cell prints the skip reason and runs nothing. That is
the intended outcome when KIM-I did not beat the same-session A160r control.

In [ ]:
if not PROCEED:
    print('SKIP KIM0: batch-1 gate failed or artifacts missing. '
          'No zero-knowledge control is run.', flush=True)
else:
    t0 = time.time()
    for i, r in enumerate(RUNS, 1):
        print(f'=== [{i}/{len(RUNS)}] {r["tag"]} '
              f'(elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
        ok = False
        try:
            ok = train_one(r)
        except Exception as e:                       # noqa: BLE001
            print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
        if ok:
            try:
                dump_probs(r)
            except Exception as e:                   # noqa: BLE001
                print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
        try:
            zip_small()
        except Exception as e:                       # noqa: BLE001
            print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
        if not ok:
            print('!! stopping KIM0 after the first failed run; fix and re-run to resume',
                  flush=True)
            break
    print(f'KIM0 finished in {(time.time() - t0) / 3600:.2f} h', flush=True)

## 5. Summary: architecture vs knowledge

Three comparisons matter:

1. `KIM0 - A160r` = value of the local-inference block with **zero** knowledge.
2. `KIMI - KIM0` = value of the five WordNet features with the **same**
   architecture.
3. `KIMI - A160r` = the headline KIM-I result (the batch-1 gate).

If `KIM0` captures the whole KIMI gain, KIM-I is an attention-residual effect,
not knowledge injection. In that case KIM-A is not justified either. If
`KIMI - KIM0` is positive with a CI above zero, the WordNet path is doing real
work; then, and only then, consider a KIM-A implementation.

In [ ]:
if not PROCEED:
    print('No KIM0 summary: gate did not pass.', flush=True)
else:
    from sklearn.metrics import roc_auc_score

    labs = np.load('weights/val_labels.npy')
    langs = np.load('weights/val_langs.npy', allow_pickle=True).astype(str)
    quads = np.load('weights/val_quads.npy', allow_pickle=True).astype(str)

    # Load batch-1 arrays again (they may live under /kaggle/input).
    for arm in ('A160r', 'KIMI'):
        for s in SEEDS:
            p, path = load_batch1_probs(f'{arm}_s{s}')
            if p is not None:
                probs_by_tag[f'{arm}_s{s}'] = p
    for s in SEEDS:
        p = f'weights/KIM0_s{s}_val_probs.npy'
        if os.path.exists(p):
            probs_by_tag[f'KIM0_s{s}'] = np.load(p)

    rows = []
    for arm in ('A160r', 'KIMI', 'KIM0'):
        for s in SEEDS:
            tag = f'{arm}_s{s}'
            if tag not in probs_by_tag:
                continue
            probs = probs_by_tag[tag]
            m = tx.metrics_from(probs.argmax(1).tolist(), labs.tolist())
            hp = f'weights/{tag}_history.csv'
            if not os.path.exists(hp):
                hp = find_artifact(f'{tag}_history.csv')
            hb = float(pd.read_csv(hp)['val_macro_f1'].max()) if hp else float(m['macro_f1'])
            rows.append(dict(arm=arm, tag=tag, seed=s, best_hist_f1=hb,
                             reload_f1=float(m['macro_f1']), reload_acc=float(m['acc']),
                             auc=float(roc_auc_score(labs, probs[:, 1])),
                             pred_pos=float((probs.argmax(1) == 1).mean())))
    summ = pd.DataFrame(rows).sort_values(['arm', 'seed'])
    summ.to_csv('weights/taskA_kim_batch2_summary.csv', index=False)
    print(summ.to_string(index=False), flush=True)

    arm_rows = []
    ens2 = {}
    for arm in ('A160r', 'KIMI', 'KIM0'):
        tags = [f'{arm}_s{s}' for s in SEEDS]
        tags = [t for t in tags if t in probs_by_tag]
        if len(tags) != 3:
            print(f'{arm}: {len(tags)}/3 seeds; skipped', flush=True)
            continue
        ens = np.mean([probs_by_tag[t] for t in tags], axis=0)
        ens2[arm] = ens
        m = tx.metrics_from(ens.argmax(1).tolist(), labs.tolist())
        arm_rows.append(dict(arm=arm, mean_best=float(summ[summ.arm == arm]['best_hist_f1'].mean()),
                             ensemble_f1=float(m['macro_f1']),
                             ensemble_acc=float(m['acc']),
                             ensemble_auc=float(roc_auc_score(labs, ens[:, 1])),
                             pred_pos=float((ens.argmax(1) == 1).mean())))
        np.save(f'weights/taskA_kim_batch2_{arm}_ensemble_probs.npy', ens)
        print(f"\n{arm}: mean best {arm_rows[-1]['mean_best']:.4f} | "
              f"ensemble {m['macro_f1']:.4f} | AUC {arm_rows[-1]['ensemble_auc']:.4f} | "
              f"pos {arm_rows[-1]['pred_pos']:.3f}", flush=True)
        for lang in sorted(set(langs.tolist())):
            msk = langs == lang
            ml = tx.metrics_from(ens[msk].argmax(1).tolist(), labs[msk].tolist())
            print(f"  [{lang}] n {int(msk.sum())} | macro_f1 {ml['macro_f1']:.4f}", flush=True)

    arm_summ = pd.DataFrame(arm_rows)
    arm_summ.to_csv('weights/taskA_kim_batch2_arms.csv', index=False)
    print()
    print(arm_summ.to_string(index=False), flush=True)

    boot = {}
    if 'KIM0' in ens2 and 'A160r' in ens2:
        boot['KIM0_vs_A160r'] = paired_cluster_bootstrap(
            labs, ens2['KIM0'], ens2['A160r'], n_boot=5000, seed=12355)
    if 'KIM0' in ens2 and 'KIMI' in ens2:
        boot['KIMI_vs_KIM0'] = paired_cluster_bootstrap(
            labs, ens2['KIMI'], ens2['KIM0'], n_boot=5000, seed=12356)
    for key, val in boot.items():
        print(f'{key}: {json.dumps(val)}', flush=True)
    with open('weights/taskA_kim_batch2_bootstrap.json', 'w', encoding='utf-8') as fh:
        json.dump(boot, fh, indent=2)

    if 'KIM0' in ens2 and 'A160r' in ens2 and 'KIMI' in ens2:
        li_gain = arm_summ.loc[arm_summ.arm == 'KIM0', 'ensemble_f1'].iloc[0] - \
            arm_summ.loc[arm_summ.arm == 'A160r', 'ensemble_f1'].iloc[0]
        kg = arm_summ.loc[arm_summ.arm == 'KIMI', 'ensemble_f1'].iloc[0] - \
            arm_summ.loc[arm_summ.arm == 'KIM0', 'ensemble_f1'].iloc[0]
        kg_ci_low = boot['KIMI_vs_KIM0']['ci_low']
        print('\n--- DECISION ---', flush=True)
        print(f'LI architecture (KIM0 - A160r): {li_gain:+.4f}', flush=True)
        print(f'knowledge (KIMI - KIM0):       {kg:+.4f} | CI low {kg_ci_low:+.4f}', flush=True)
        if kg >= 0.005 and kg_ci_low > 0:
            print('The KIM-I gain is knowledge-specific. Keep KIM-I. '
                  'KIM-A is the only justified follow-up, but it still needs an implementation.',
                  flush=True)
        elif abs(kg) < 0.005 or kg_ci_low <= 0:
            print('No knowledge-specific gain: the local-inference block, not WordNet, explains '
                  'the KIM-I result. Do not implement KIM-A.', flush=True)
        else:
            print('The zero-knowledge control is better than KIM-I: the WordNet features hurt. '
                  'Stop this line.', flush=True)
        with open('weights/taskA_kim_batch2_decision.json', 'w', encoding='utf-8') as fh:
            json.dump(dict(li_gain=float(li_gain), knowledge_gain=float(kg),
                           knowledge_ci_low=float(kg_ci_low)), fh, indent=2)

    zip_small()
    print('TASK A KIM BATCH2 COMPLETE', flush=True)